# StoichiometryMode Test

 > Suite role: mode-centric validation and comparison of all three stoichiometric modes under identical conditions.

Tests the three stoichiometric model implementations:
- `StoichiometryMode.CONSTANT` — yields fixed at mu_max, no recalculation during simulation
- `StoichiometryMode.DYNAMIC` — yields recalculated at the current growth rate each ODE step
- `StoichiometryMode.EXCESS_COUPLED` — catabolic stoichiometry updated from excess-model fluxes each step

**Test Coverage:**
1. Instantiation of all three stoichiometric modes (`CONSTANT`, `DYNAMIC`, `EXCESS_COUPLED`)
2. Unit test of `calculate_yields` behaviour in all modes (including substrate-dependent catabolic yields for `EXCESS_COUPLED`)
3. Edge cases: `mu = 0`, `mu < 0`, `mu = mu_max`
4. Yield variation with mu across a range of growth rates (`DYNAMIC` and `EXCESS_COUPLED`)
5. End-to-end batch simulation in both analytical modes — verify results agree closely
6. Performance benchmark: CONSTANT vs DYNAMIC overhead
7. Three-mode comparison (`CONSTANT`, `DYNAMIC`, `EXCESS_COUPLED`) for q-rates and stoichiometric yields

## 1. Imports

In [1]:
import sys
import time
from pathlib import Path

# Add src to path
repo_root = Path.cwd().parent
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from animakinetica.core.parameters import SimulationParameters
from animakinetica.core.simulation import CellCultureSimulation
from animakinetica.models.stoichiometry import StoichiometricModel, StoichiometryMode
from animakinetica.models.kinetics import Kinetics, create_kinetic_model
from animakinetica.models.odesystems import OperatingMode

print("✓ Imports successful")

✓ Imports successful


## 2. Load Parameters

In [2]:
params_sto = SimulationParameters.load_parameters([
    "base_parameters",
    "stoichiometry/anab_GFI",
    "stoichiometry/cata_GFI",
    "cell_properties/cell_GFI",
])

params_exc = SimulationParameters.load_parameters([
    "base_parameters",
    "excess_kinetics/excess_kinetics_batch",
    "stoichiometry/anab_GFI",
   # "stoichiometry/cata_GFI",
    "cell_properties/cell_GFI",
])

# Use stoichiometric parameter set as baseline reference for mu_max
params = params_sto
mu_max = float(params.cell_properties.mu_max)
print("✓ Parameters loaded")
print(f"  STO params  : mu_max = {mu_max:.5f} h⁻¹  (td = {params.cell_properties.td.value:.1f} h)")
print("  EXCESS params: base + excess_kinetics + anabolism + cell properties")

Loading parameters from 4 file(s) sequentially...
  [1/4] Base: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\base_parameters.yaml
  [2/4] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\stoichiometry\anab_GFI.yaml
  [3/4] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\stoichiometry\cata_GFI.yaml
  [4/4] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\cell_properties\cell_GFI.yaml
Successfully loaded parameters from 4 file(s).
Loading parameters from 4 file(s) sequentially...
  [1/4] Base: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\base_parameters.yaml
  [2/4] Updat

## 3. Instantiation of All Three Modes

Verify all stoichiometric modes initialise correctly and that `self.mode` is set as expected.

In [3]:
model_const  = StoichiometricModel(params_sto, mode=StoichiometryMode.CONSTANT)
model_dyn    = StoichiometricModel(params_sto, mode=StoichiometryMode.DYNAMIC)
model_exc    = StoichiometricModel(params_exc, mode=StoichiometryMode.EXCESS_COUPLED)
model_default = StoichiometricModel(params_sto)  # should default to CONSTANT

assert model_const.mode   == StoichiometryMode.CONSTANT, "CONSTANT mode not set correctly"
assert model_dyn.mode     == StoichiometryMode.DYNAMIC,  "DYNAMIC mode not set correctly"
assert model_exc.mode     == StoichiometryMode.EXCESS_COUPLED, "EXCESS_COUPLED mode not set correctly"
assert model_default.mode == StoichiometryMode.CONSTANT, "Default mode should be CONSTANT"

# Yields must be pre-calculated and non-None after __init__ for all modes
assert model_const.yields is not None, "CONSTANT: yields should be pre-calculated"
assert model_dyn.yields   is not None, "DYNAMIC:  yields should be pre-calculated"
assert model_exc.yields   is not None, "EXCESS_COUPLED: yields placeholder should be initialized"
assert len(model_const.yields) == 8
assert len(model_dyn.yields)   == 8
assert len(model_exc.yields)   == 8

print("✓ All three models instantiate correctly")
print(f"  CONSTANT mode      : {model_const.mode}")
print(f"  DYNAMIC mode       : {model_dyn.mode}")
print(f"  EXCESS_COUPLED mode: {model_exc.mode}")
print(f"  Default mode       : {model_default.mode}")
print(f"\n  Yields shape       : {model_const.yields.shape}")
print("  Labels             : [Y_Glc/X, Y_Lac/Glc, Y_Gln/Glc, Y_O2/Glc, Y_CO2/Glc, Y_AA/Glc, Y_Amm/Gln, Y_Glu/X, Y_H2O/X]")

✓ All three models instantiate correctly
  CONSTANT mode      : StoichiometryMode.CONSTANT
  DYNAMIC mode       : StoichiometryMode.DYNAMIC
  EXCESS_COUPLED mode: StoichiometryMode.EXCESS_COUPLED
  Default mode       : StoichiometryMode.CONSTANT

  Yields shape       : (8,)
  Labels             : [Y_Glc/X, Y_Lac/Glc, Y_Gln/Glc, Y_O2/Glc, Y_CO2/Glc, Y_AA/Glc, Y_Amm/Gln, Y_Glu/X, Y_H2O/X]


## 4. Unit Test: `calculate_yields` Behaviour

**CONSTANT** — calling `calculate_yields(mu)` with any `mu` must always return exactly the cached array.  
**DYNAMIC** — calling `calculate_yields(mu)` with a different `mu` must return updated values.  
**EXCESS_COUPLED** — calling `calculate_yields(mu, cat_yields=...)` must accept catabolic yields derived from excess kinetics at specified substrate concentrations.

In [4]:
mu_test_values = [mu_max * 0.1, mu_max * 0.5, mu_max, mu_max * 0.9]

# Excess kinetics model for generating catabolic yield inputs to EXCESS_COUPLED mode
kin_exc = create_kinetic_model(Kinetics.EXCESS, params_exc, stoichiometric_model=model_exc)

# --- CONSTANT mode: yields must be identical regardless of mu ---
baseline_const = model_const.yields.copy()
for mu in mu_test_values:
    result = model_const.calculate_yields(mu)
    assert np.array_equal(result, baseline_const), (
        f"CONSTANT: yields changed at mu={mu:.4f} — expected no recalculation"
    )
print("✓ CONSTANT mode: yields unchanged across all mu values (correct)")

# --- DYNAMIC mode: yields should vary with mu ---
yields_at_mu = {}
for mu in mu_test_values:
    yields_at_mu[mu] = model_dyn.calculate_yields(mu).copy()

# Y_Glc/X (index 0) decreases as mu increases (less maintenance overhead per unit growth)
y_glc_x = [yields_at_mu[mu][0] for mu in sorted(mu_test_values)]
assert all(y_glc_x[i] >= y_glc_x[i+1] for i in range(len(y_glc_x)-1)), (
    "DYNAMIC: Y_Glc/X should decrease as mu increases"
)
print("✓ DYNAMIC mode: Y_Glc/X decreases as mu increases (correct)")

# At mu = mu_max the DYNAMIC result should match CONSTANT (both pre-calculated at mu_max)
yields_dyn_at_mumax = model_dyn.calculate_yields(mu_max)
np.testing.assert_allclose(
    yields_dyn_at_mumax, baseline_const, rtol=1e-10,
    err_msg="At mu=mu_max, DYNAMIC and CONSTANT yields should be identical"
)
print("✓ At mu=mu_max, DYNAMIC yields match CONSTANT yields (correct)")

# --- EXCESS_COUPLED mode: requires cat_yields from excess kinetics + substrate concentrations ---
excess_test_points = [
    (mu_max * 0.2, 0.1, 0.04),
    (mu_max * 0.75, 5.0, 1.0),
    (mu_max * 0.9, 20.0, 3.0),
]

for mu, c_glc, c_gln in excess_test_points:
    cat_yields = kin_exc.calculate_cat_yields_from_excess(mu, c_glc, c_gln)
    y_exc = model_exc.calculate_yields(mu, cat_yields=cat_yields)
    assert y_exc is not None and len(y_exc) == 8, f"EXCESS_COUPLED failed for mu={mu:.4f}"
    assert not np.any(np.isnan(y_exc)), f"EXCESS_COUPLED returned NaN for mu={mu:.4f}"

# Check that changing substrate concentrations at fixed mu changes catabolic-dependent yields
mu_ref = mu_max * 0.6
cat_low = kin_exc.calculate_cat_yields_from_excess(mu_ref, C_Glc=2.0, C_Gln=0.3)
cat_high = kin_exc.calculate_cat_yields_from_excess(mu_ref, C_Glc=25.0, C_Gln=5.0)
y_low = model_exc.calculate_yields(mu_ref, cat_yields=cat_low).copy()
y_high = model_exc.calculate_yields(mu_ref, cat_yields=cat_high).copy()
assert not np.allclose(y_low[1:5], y_high[1:5], rtol=1e-6, atol=1e-10), (
    "EXCESS_COUPLED: catabolic-dependent yields should change when substrate concentrations change"
)
print("✓ EXCESS_COUPLED mode accepts substrate-driven catabolic yields and responds to substrate changes (correct)")

✓ CONSTANT mode: yields unchanged across all mu values (correct)
✓ DYNAMIC mode: Y_Glc/X decreases as mu increases (correct)
✓ At mu=mu_max, DYNAMIC yields match CONSTANT yields (correct)
✓ EXCESS_COUPLED mode accepts substrate-driven catabolic yields and responds to substrate changes (correct)


## Sweep

In [5]:
logged_terms = [
    "q_Lac_lim",
    "q_Lac_exc",
    "q_Glc_lim",
    "q_Glc_ana",
    "q_Glc_cat_lim",
    "delta_q_Glc_Glc",
    "delta_q_Gln_Glc",
    "delta_q_Glc_Gln",
    "delta_q_Gln_Gln",
    "q_Glc_exc",
    "q_Gln_exc",
    "q_Glc_cat",
    "q_Gln_cat",
    "q_Lac_cat",
    "Y_Lac_Glc_cat",
    "Y_Gln_Glc_cat",
]

def sweep_cat_yield_terms(
    model,
    sweep_var,
    sweep_values,
    *,
    mu=None,
    C_Glc=None,
    C_Gln=None,
):
    fixed_inputs = {"mu": mu, "C_Glc": C_Glc, "C_Gln": C_Gln}
    if sweep_var not in fixed_inputs:
        raise ValueError(f"Unknown sweep variable: {sweep_var}")

    rows = []
    for value in sweep_values:
        inputs = fixed_inputs.copy()
        inputs[sweep_var] = float(value)
        terms = model.get_cat_yield_terms(**inputs)
        rows.append({**inputs, **terms})

    return pd.DataFrame(rows)

def plot_cat_yield_sweep(df, x_col, title, color):
    positions = [
        (1, 1), (1, 2), (1, 3), (1, 4),
        (2, 1), (2, 2), (2, 3), (2, 4),
        (3, 1), (3, 2), (3, 3), (3, 4),
        (4, 1), (4, 2), (4, 3), (4, 4),
    ]

    fig = make_subplots(
        rows=4,
        cols=4,
        subplot_titles=logged_terms,
        horizontal_spacing=0.08,
        vertical_spacing=0.09,
    )

    for term, (row, col) in zip(logged_terms, positions):
        fig.add_trace(
            go.Scatter(
                x=df[x_col],
                y=df[term],
                mode="lines",
                line=dict(color=color, width=2),
                showlegend=False,
                name=term,
            ),
            row=row,
            col=col,
        )
        fig.update_yaxes(title_text=term, row=row, col=col)

    fig.update_xaxes(title_text=x_col)
    fig.update_layout(
        title_text=title,
        height=1400,
        width=1400,
        template="plotly_white",
        margin=dict(t=100, l=40, r=40, b=40),
    )
    return fig

mu_fixed = mu_max
C_Glc_fixed = 1000
C_Gln_fixed = 100

glc_sweep = sweep_cat_yield_terms(
    kin_exc,
    "C_Glc",
    np.linspace(0.0, 1000.0, 200),
    mu=mu_fixed,
    C_Gln=C_Gln_fixed,
)

gln_sweep = sweep_cat_yield_terms(
    kin_exc,
    "C_Gln",
    np.linspace(0.0, 50.0, 200),
    mu=mu_fixed,
    C_Glc=C_Glc_fixed,
)

mu_sweep = sweep_cat_yield_terms(
    kin_exc,
    "mu",
    np.linspace(0.0, mu_max, 200),
    C_Glc=C_Glc_fixed,
    C_Gln=C_Gln_fixed,
)

preview_cols = ["mu", "C_Glc", "C_Gln"] + logged_terms
display(glc_sweep[preview_cols].head())

fig_glc = plot_cat_yield_sweep(
    glc_sweep,
    "C_Glc",
    f"Excess catabolic terms vs glucose concentration at mu={mu_fixed:.4f} h^-1 and C_Gln={C_Gln_fixed:.2f} mmol/L",
    color="firebrick",
)
fig_gln = plot_cat_yield_sweep(
    gln_sweep,
    "C_Gln",
    f"Excess catabolic terms vs glutamine concentration at mu={mu_fixed:.4f} h^-1 and C_Glc={C_Glc_fixed:.2f} mmol/L",
    color="darkgreen",
)
fig_mu = plot_cat_yield_sweep(
    mu_sweep,
    "mu",
    f"Excess catabolic terms vs growth rate at C_Glc={C_Glc_fixed:.2f} mmol/L and C_Gln={C_Gln_fixed:.2f} mmol/L",
    color="royalblue",
)

fig_glc.show()
fig_gln.show()
fig_mu.show()

,mu,C_Glc,C_Gln,q_Lac_lim,q_Lac_exc,q_Glc_lim,q_Glc_ana,q_Glc_cat_lim,delta_q_Glc_Glc,delta_q_Gln_Glc,delta_q_Glc_Gln,delta_q_Gln_Gln,q_Glc_exc,q_Gln_exc,q_Glc_cat,q_Gln_cat,q_Lac_cat,Y_Lac_Glc_cat,Y_Gln_Glc_cat
0,0.038508,0.000000,100,2.037637e-11,0.000000e+00,6.112409e-11,2.037135e-11,4.075274e-11,0.000000e+00,2.293798e-11,0.0,2.998199e-11,2.293798e-11,2.998199e-11,6.369072e-11,2.998199e-11,2.037637e-11,0.319927,0.470743
1,0.038508,5.025126,100,2.037637e-11,1.767495e-10,6.112409e-11,2.037135e-11,4.075274e-11,7.490710e-11,2.293798e-11,0.0,2.998199e-11,9.784508e-11,2.998199e-11,1.385978e-10,2.998199e-11,1.971259e-10,1.422287,0.216324
2,0.038508,10.050251,100,2.037637e-11,2.475290e-10,6.112409e-11,2.037135e-11,4.075274e-11,1.049037e-10,2.293798e-11,0.0,2.998199e-11,1.278417e-10,2.998199e-11,1.685944e-10,2.998199e-11,2.679054e-10,1.589052,0.177835
3,0.038508,15.075377,100,2.037637e-11,2.843625e-10,6.112409e-11,2.037135e-11,4.075274e-11,1.205139e-10,2.293798e-11,0.0,2.998199e-11,1.434519e-10,2.998199e-11,1.842046e-10,2.998199e-11,3.047389e-10,1.654350,0.162765
4,0.038508,20.100503,100,2.037637e-11,3.069474e-10,6.112409e-11,2.037135e-11,4.075274e-11,1.300855e-10,2.293798e-11,0.0,2.998199e-11,1.530234e-10,2.998199e-11,1.937762e-10,2.998199e-11,3.273238e-10,1.689185,0.154725


## 5. Edge Cases: `mu ≤ 0`

All modes must handle `mu = 0` and `mu < 0` safely without raising errors (guards in `calculate_yields`).

For `EXCESS_COUPLED`, catabolic yields are still supplied from excess kinetics at a reference substrate condition.

In [6]:
# Prime EXCESS_COUPLED with one valid positive-mu state so fallback behaviour has a reference
mu_prime = mu_max * 0.6
cat_prime = kin_exc.calculate_cat_yields_from_excess(mu_prime, C_Glc=10.0, C_Gln=1.5)
_ = model_exc.calculate_yields(mu_prime, cat_yields=cat_prime)

for mu_edge in [0.0, -0.001, -1.0]:
    result_const = model_const.calculate_yields(mu_edge)
    result_dyn   = model_dyn.calculate_yields(mu_edge)
    result_exc   = model_exc.calculate_yields(mu_edge, cat_yields=cat_prime)

    assert result_const is not None and len(result_const) == 8, f"CONSTANT failed for mu={mu_edge}"
    assert result_dyn   is not None and len(result_dyn)   == 8, f"DYNAMIC failed for mu={mu_edge}"
    assert result_exc   is not None and len(result_exc)   == 8, f"EXCESS_COUPLED failed for mu={mu_edge}"

    assert not np.any(np.isnan(result_const)), f"CONSTANT returned NaN for mu={mu_edge}"
    assert not np.any(np.isnan(result_dyn)),   f"DYNAMIC returned NaN for mu={mu_edge}"
    assert not np.any(np.isnan(result_exc)),   f"EXCESS_COUPLED returned NaN for mu={mu_edge}"

print("✓ All modes handle mu ≤ 0 safely (no NaN, no exception)")
print(f"  DYNAMIC at mu=0 fallback yields       : {model_dyn.calculate_yields(0.0).round(4)}")
print(f"  EXCESS_COUPLED at mu=0 fallback yields: {model_exc.calculate_yields(0.0, cat_yields=cat_prime).round(4)}")

✓ All modes handle mu ≤ 0 safely (no NaN, no exception)
  DYNAMIC at mu=0 fallback yields       : [0.2575 0.5829 0.2934 2.888  3.202  0.2889 0.655  0.9488]
  EXCESS_COUPLED at mu=0 fallback yields: [0. 0. 0. 0. 0. 0. 0. 0.]


## 6.1 Yield Variation with mu

Plot how each yield coefficient changes across the full range of biologically relevant growth rates.

Comparison includes:
- `DYNAMIC` (mu-dependent stoichiometric recalculation)
- `EXCESS_COUPLED` (mu + substrate-dependent catabolic yield update) at fixed substrate concentrations
- `CONSTANT` baseline (horizontal dashed line)

In [7]:
mu_range = np.linspace(mu_max * 0.01, mu_max, 200)
yield_labels = [
    "Y_Glc/X", "Y_Lac/Glc", "Y_Gln/Glc", "Y_O2/Glc", "Y_CO2/Glc",
    "Y_AA/Glc", "Y_Amm/Gln", "Y_H2O/X",
]

# DYNAMIC mode yield curve (mu-dependent only)
yields_dyn_matrix = np.array([model_dyn.calculate_yields(mu) for mu in mu_range])

# EXCESS_COUPLED mode yield curve (mu + substrate-dependent catabolic yields)
C_Glc_exc_plot = 1.0
C_Gln_exc_plot = 0.2
yields_exc_matrix = np.array([
    model_exc.calculate_yields(
        mu,
        cat_yields=kin_exc.calculate_cat_yields_from_excess(mu, C_Glc_exc_plot, C_Gln_exc_plot),
    )
    for mu in mu_range
])

# CONSTANT baseline for comparison
const_yields = model_const.yields

fig = make_subplots(
    rows=5, cols=2,
    shared_xaxes=False,
)

positions = [(1,1),(1,2),(2,1),(2,2),(3,1),(3,2),(4,1),(4,2)]

for i, (label, (row, col)) in enumerate(zip(yield_labels, positions)):

    fig.add_trace(go.Scatter(
        x=mu_range, y=np.full_like(mu_range, const_yields[i]),
        mode='lines', name="CONSTANT (dashed red)",
        line=dict(color='firebrick', width=2, dash='dash'),
        showlegend=(i == 0),
    ), row=row, col=col)

    fig.add_trace(go.Scatter(
        x=mu_range, y=yields_dyn_matrix[:, i],
        mode='lines', name="DYNAMIC (solid blue)",
        line=dict(color='royalblue', width=2),
        showlegend=(i == 0),
    ), row=row, col=col)

    fig.add_trace(go.Scatter(
        x=mu_range, y=yields_exc_matrix[:, i],
        mode='lines', name="EXCESS_COUPLED (solid green)",
        line=dict(color='darkgreen', width=2),
        showlegend=(i == 0),
    ), row=row, col=col)

    fig.update_yaxes(title_text=label, row=row, col=col)

    if label == "Y_Glc/X":
        fig.update_yaxes(range=[0, 2], row=row, col=col)

fig.update_xaxes(title_text="μ (h⁻¹)")
fig.update_layout(
    title_text="Yield coefficients vs growth rate across stoichiometry modes",
    height=1900,
    width=1000,
    template="plotly_white",
    legend=dict(orientation='v', yanchor='bottom', y=0.1, xanchor='left', x=0.6),
    margin=dict(t=140),
    annotations=list(fig.layout.annotations) + [
        dict(
            text=f"EXCESS_COUPLED fixed substrates: C_Glc={C_Glc_exc_plot:.1f} mmol/L, C_Gln={C_Gln_exc_plot:.1f} mmol/L",
            x=0.5, y=1.02, xref='paper', yref='paper',
            showarrow=False, font=dict(size=13, color='black')
        )
    ],
)
fig.show()

## 6.2 Yield Variation with Glc concentration

Plot how each yield coefficient changes across the full range of biologically relevant Glucose concentration.

In [8]:
yield_labels = [
    "Y_Glc/X", "Y_Lac/Glc", "Y_Gln/Glc", "Y_O2/Glc", "Y_CO2/Glc",
    "Y_AA/Glc", "Y_Amm/Gln", "Y_H2O/X",
]

# EXCESS_COUPLED mode yield curve (mu + substrate-dependent catabolic yields)
Glc_range = np.linspace(0, 50, 200) # mmol/L
Gln_fixed = 1.0 # mmol/L
mu_fixed = mu_max * 0.8
yields_exc_matrix = np.array([
    model_exc.calculate_yields(
        mu_fixed,
        cat_yields=kin_exc.calculate_cat_yields_from_excess(mu_fixed, Glc, Gln_fixed),
    )
    for Glc in Glc_range
])

# CONSTANT baseline for comparison
const_yields = model_const.yields
# DYNAMIC mode yield curve (mu-dependent only, so constant across Glc range)
dynamic_yields = model_dyn.calculate_yields(mu_fixed)

fig = make_subplots(
    rows=5, cols=2,
    shared_xaxes=False,
)

positions = [(1,1),(1,2),(2,1),(2,2),(3,1),(3,2),(4,1),(4,2),(5,1)]

for i, (label, (row, col)) in enumerate(zip(yield_labels, positions)):
    fig.add_trace(go.Scatter(
        x=Glc_range, y=np.full_like(Glc_range, dynamic_yields[i]),
        mode='lines', name="DYNAMIC (solid blue)",
        line=dict(color='royalblue', width=2),
        showlegend=(i == 0),
    ), row=row, col=col)

    fig.add_trace(go.Scatter(
        x=Glc_range, y=yields_exc_matrix[:, i],
        mode='lines', name="EXCESS_COUPLED (solid green)",
        line=dict(color='darkgreen', width=2),
        showlegend=(i == 0),
    ), row=row, col=col)

    fig.add_trace(go.Scatter(
        x=Glc_range, y=np.full_like(Glc_range, const_yields[i]),
        mode='lines', name="CONSTANT (dashed red)",
        line=dict(color='firebrick', width=2, dash='dash'),
        showlegend=(i == 0),
    ), row=row, col=col)

    fig.update_yaxes(title_text=label, row=row, col=col)

fig.update_xaxes(title_text="Glc (mmol/L)")
fig.update_layout(
    title_text="Yield coefficients vs Glucose concentration across stoichiometry modes",
    height=1900,
    width=1000,
    template="plotly_white",
    legend=dict(orientation='v', yanchor='bottom', y=0.1, xanchor='left', x=0.6),
    margin=dict(t=140),
    annotations=list(fig.layout.annotations) + [
        dict(
            text=f"Fixed paramters: mu={mu_fixed:.3f} h-1, C_Gln={Gln_fixed:.1f} mmol/L",
            x=0.5, y=1.02, xref='paper', yref='paper',
            showarrow=False, font=dict(size=13, color='black')
        )
    ],
)
fig.show()

In [9]:
yield_labels = [
    "Y_Glc/X", "Y_Lac/Glc", "Y_Gln/Glc", "Y_O2/Glc", "Y_CO2/Glc",
    "Y_AA/Glc", "Y_Amm/Gln", "Y_H2O/X",
]

# EXCESS_COUPLED mode yield curve (mu + substrate-dependent catabolic yields)
Gln_range = np.linspace(0, 50, 200) # mmol/L
Glc_fixed = 5.0 # mmol/L
mu_fixed = mu_max * 0.8
yields_exc_matrix = np.array([
    model_exc.calculate_yields(
        mu_fixed,
        cat_yields=kin_exc.calculate_cat_yields_from_excess(mu_fixed, Glc_fixed, Gln),
    )
    for Gln in Gln_range
])

# CONSTANT baseline for comparison
const_yields = model_const.yields
# DYNAMIC mode yield curve (mu-dependent only, so constant across Glc range)
dynamic_yields = model_dyn.calculate_yields(mu_fixed)

fig = make_subplots(
    rows=5, cols=2,
    shared_xaxes=False,
)

positions = [(1,1),(1,2),(2,1),(2,2),(3,1),(3,2),(4,1),(4,2)]

for i, (label, (row, col)) in enumerate(zip(yield_labels, positions)):
    fig.add_trace(go.Scatter(
        x=Gln_range, y=np.full_like(Gln_range, dynamic_yields[i]),
        mode='lines', name="DYNAMIC (solid blue)",
        line=dict(color='royalblue', width=2),
        showlegend=(i == 0),
    ), row=row, col=col)

    fig.add_trace(go.Scatter(
        x=Gln_range, y=yields_exc_matrix[:, i],
        mode='lines', name="EXCESS_COUPLED (solid green)",
        line=dict(color='darkgreen', width=2),
        showlegend=(i == 0),
    ), row=row, col=col)

    fig.add_trace(go.Scatter(
        x=Gln_range, y=np.full_like(Gln_range, const_yields[i]),
        mode='lines', name="CONSTANT (dashed red)",
        line=dict(color='firebrick', width=2, dash='dash'),
        showlegend=(i == 0),
    ), row=row, col=col)

    fig.update_yaxes(title_text=label, row=row, col=col)

fig.update_xaxes(title_text="Gln (mmol/L)")
fig.update_layout(
    title_text="Yield coefficients vs Glutamine concentration across stoichiometry modes",
    height=1900,
    width=1000,
    template="plotly_white",
    legend=dict(orientation='v', yanchor='bottom', y=0.1, xanchor='left', x=0.6),
    margin=dict(t=140),
    annotations=list(fig.layout.annotations) + [
        dict(
            text=f"Fixed paramters: mu={mu_fixed:.3f} h-1, C_Glc={Glc_fixed:.1f} mmol/L",
            x=0.5, y=1.02, xref='paper', yref='paper',
            showarrow=False, font=dict(size=13, color='black')
        )
    ],
)
fig.show()

## 7. End-to-End Batch Simulation: CONSTANT vs DYNAMIC vs EXCESS_COUPLED

Run full batch simulations in all three modes and compare key state variable trajectories.
This checks whether EXCESS_COUPLED remains numerically well-behaved and how strongly it diverges from the stoichiometric modes under identical initial conditions.

In [10]:
sim_const = CellCultureSimulation(
    parameters=params_sto,
    kinetics=Kinetics.STOICHIOMETRIC,
    operating_mode=OperatingMode.BATCH,
    stoichiometry_mode=StoichiometryMode.CONSTANT,
    performance_mode=False,
)
result_const = sim_const.run_simulation(number_of_batches=1)
batch_const = result_const.batches[0]
print(f"CONSTANT       - Duration: {batch_const.t_end:.1f} h | Xv_final: {batch_const.Xv_final:.3e} cells/L")

sim_dyn = CellCultureSimulation(
    parameters=params_sto,
    kinetics=Kinetics.STOICHIOMETRIC,
    operating_mode=OperatingMode.BATCH,
    stoichiometry_mode=StoichiometryMode.DYNAMIC,
    performance_mode=False,
)
result_dyn = sim_dyn.run_simulation(number_of_batches=1)
batch_dyn = result_dyn.batches[0]
print(f"DYNAMIC        - Duration: {batch_dyn.t_end:.1f} h | Xv_final: {batch_dyn.Xv_final:.3e} cells/L")

sim_exc = CellCultureSimulation(
    parameters=params_exc,
    kinetics=Kinetics.EXCESS,
    operating_mode=OperatingMode.BATCH,
    stoichiometry_mode=StoichiometryMode.EXCESS_COUPLED,
    performance_mode=False,
)
result_exc = sim_exc.run_simulation(number_of_batches=1)
batch_exc = result_exc.batches[0]
print(f"EXCESS_COUPLED - Duration: {batch_exc.t_end:.1f} h | Xv_final: {batch_exc.Xv_final:.3e} cells/L")

run_simulation execution time: 50.259 ms
CONSTANT       - Duration: 97.0 h | Xv_final: 1.039e+10 cells/L
run_simulation execution time: 40.494 ms
DYNAMIC        - Duration: 99.9 h | Xv_final: 1.039e+10 cells/L
run_simulation execution time: 45.670 ms
EXCESS_COUPLED - Duration: 82.0 h | Xv_final: 4.722e+09 cells/L


In [11]:
# Interpolate all solutions onto a common time grid for comparison
t_common = np.linspace(0, min(batch_const.t_end, batch_dyn.t_end, batch_exc.t_end), 500)

# State variable indices: Xv=0, C_Glc=1, C_Gln=2, C_Lac=3, C_Amm=4
state_labels  = ["Xv (cells/L)", "C_Glc (mmol/L)", "C_Gln (mmol/L)", "C_Lac (mmol/L)", "C_Amm (mmol/L)"]
state_indices = [0, 1, 2, 3, 4]

fig = make_subplots(rows=2, cols=3, subplot_titles=state_labels)
positions = [(1,1), (1,2), (1,3), (2,1), (2,2)]

for i, (label, (row, col)) in enumerate(zip(state_labels, positions)):
    idx = state_indices[i]
    y_const = np.interp(t_common, batch_const.t_points, batch_const.y_points[idx, :])
    y_dyn = np.interp(t_common, batch_dyn.t_points, batch_dyn.y_points[idx, :])
    y_exc = np.interp(t_common, batch_exc.t_points, batch_exc.y_points[idx, :])

    fig.add_trace(
        go.Scatter(
            x=t_common, y=y_const, mode='lines', name='CONSTANT',
            line=dict(color='firebrick', width=2, dash='dash'),
            showlegend=(i == 0),
        ),
        row=row, col=col,
    )
    fig.add_trace(
        go.Scatter(
            x=t_common, y=y_dyn, mode='lines', name='DYNAMIC',
            line=dict(color='royalblue', width=2),
            showlegend=(i == 0),
        ),
        row=row, col=col,
    )
    fig.add_trace(
        go.Scatter(
            x=t_common, y=y_exc, mode='lines', name='EXCESS_COUPLED',
            line=dict(color='darkgreen', width=2),
            showlegend=(i == 0),
        ),
        row=row, col=col,
    )

fig.update_xaxes(title_text='Time (h)')
fig.update_layout(
    title_text='Batch simulation: CONSTANT vs DYNAMIC vs EXCESS_COUPLED',
    height=500,
    template='plotly_white',
    legend=dict(x=0.66, y=0.25),
)
fig.show()

In [12]:
# Numerical comparison of final state metrics across all three modes
print('Final state comparison')
print('=' * 88)
headers = ['Variable', 'CONSTANT', 'DYNAMIC', 'EXCESS_COUPLED', 'Δdyn (%rel)', 'Δexc (%rel)']
print(f"{headers[0]:<22} {headers[1]:>12} {headers[2]:>12} {headers[3]:>15} {headers[4]:>12} {headers[5]:>12}")
print('-' * 88)

rows = [
    ('Xv_final (cells/L)', batch_const.Xv_final, batch_dyn.Xv_final, batch_exc.Xv_final),
    ('t_end (h)', batch_const.t_end, batch_dyn.t_end, batch_exc.t_end),
    ('C_Glc_final (mmol/L)', batch_const.y_points[1, -1], batch_dyn.y_points[1, -1], batch_exc.y_points[1, -1]),
    ('C_Lac_final (mmol/L)', batch_const.y_points[3, -1], batch_dyn.y_points[3, -1], batch_exc.y_points[3, -1]),
    ('C_Amm_final (mmol/L)', batch_const.y_points[4, -1], batch_dyn.y_points[4, -1], batch_exc.y_points[4, -1]),
]

for name, val_c, val_d, val_e in rows:
    delta_dyn = abs(val_d - val_c) / max(abs(val_c), 1e-20) * 100
    delta_exc = abs(val_e - val_c) / max(abs(val_c), 1e-20) * 100
    print(f"{name:<22} {val_c:>12.4g} {val_d:>12.4g} {val_e:>15.4g} {delta_dyn:>11.3f}% {delta_exc:>11.3f}%")

Final state comparison
Variable                   CONSTANT      DYNAMIC  EXCESS_COUPLED  Δdyn (%rel)  Δexc (%rel)
----------------------------------------------------------------------------------------
Xv_final (cells/L)        1.039e+10    1.039e+10       4.722e+09       0.000%      54.537%
t_end (h)                     97.05        99.87           82.03       2.910%      15.474%
C_Glc_final (mmol/L)          7.699        2.338           2.343      69.633%      69.564%
C_Lac_final (mmol/L)          10.09        14.55           34.73      44.218%     244.320%
C_Amm_final (mmol/L)          3.325        4.795            8.15      44.218%     145.128%


## 8. Performance Benchmark

Compare `calculate_yields` overhead across all three modes (`CONSTANT`, `DYNAMIC`, `EXCESS_COUPLED`) by timing repeated calls at the same reference condition.

In [13]:
N_CALLS = 100_000
mu_bench = mu_max * 0.5
C_Glc_bench = 10.0
C_Gln_bench = 1.5
cat_yields_bench = kin_exc.calculate_cat_yields_from_excess(mu_bench, C_Glc_bench, C_Gln_bench)

# Warm up
for _ in range(100):
    model_const.calculate_yields(mu_bench)
    model_dyn.calculate_yields(mu_bench)
    model_exc.calculate_yields(mu_bench, cat_yields=cat_yields_bench)

# Time CONSTANT
t0 = time.perf_counter()
for _ in range(N_CALLS):
    model_const.calculate_yields(mu_bench)
t_const = time.perf_counter() - t0

# Time DYNAMIC
t0 = time.perf_counter()
for _ in range(N_CALLS):
    model_dyn.calculate_yields(mu_bench)
t_dyn = time.perf_counter() - t0

# Time EXCESS_COUPLED (catabolic yields precomputed to isolate stoichiometry update cost)
t0 = time.perf_counter()
for _ in range(N_CALLS):
    model_exc.calculate_yields(mu_bench, cat_yields=cat_yields_bench)
t_exc = time.perf_counter() - t0

print(f"{'Mode':<16} {'Total (s)':>12} {'Per call (us)':>16} {'Relative':>12}")
print('-' * 60)
print(f"{'CONSTANT':<16} {t_const:>12.4f} {t_const/N_CALLS*1e6:>16.3f} {'1.0x':>12}")
print(f"{'DYNAMIC':<16} {t_dyn:>12.4f} {t_dyn/N_CALLS*1e6:>16.3f} {t_dyn/t_const:>11.1f}x")
print(f"{'EXCESS_COUPLED':<16} {t_exc:>12.4f} {t_exc/N_CALLS*1e6:>16.3f} {t_exc/t_const:>11.1f}x")
print(f"\nCalls: {N_CALLS:,}")

Mode                Total (s)    Per call (us)     Relative
------------------------------------------------------------
CONSTANT               0.1024            1.024         1.0x
DYNAMIC                0.5916            5.916         5.8x
EXCESS_COUPLED         1.0185           10.185        10.0x

Calls: 100,000


## 9.q-rates + stoichiometric yields

> Goal: provide a presentable side-by-side comparison for all three modes under the same state trajectory.

For `CONSTANT` and `DYNAMIC`, rates are computed with `Kinetics.STOICHIOMETRIC`.
For `EXCESS_COUPLED`, rates are computed with `Kinetics.EXCESS` and the catabolic stoichiometry is updated from excess fluxes.

In [14]:
sto_const = StoichiometricModel(params_sto, mode=StoichiometryMode.CONSTANT)
sto_dyn = StoichiometricModel(params_sto, mode=StoichiometryMode.DYNAMIC)
sto_exc = StoichiometricModel(params_exc, mode=StoichiometryMode.EXCESS_COUPLED)

# Reuse notebook parameter sets loaded in Section 2
kin_const = create_kinetic_model(Kinetics.STOICHIOMETRIC, params_sto, stoichiometric_model=sto_const)
kin_dyn = create_kinetic_model(Kinetics.STOICHIOMETRIC, params_sto, stoichiometric_model=sto_dyn)
kin_exc_cmp = create_kinetic_model(Kinetics.EXCESS, params_exc, stoichiometric_model=sto_exc)



def eval_point(kin_model, sto_model, params_local, t, y, c_co2):
    kin_model.osm_current = float(params_local.initial_conditions.osm_bg_init)
    kin_model.calculate_rates(t=t, y=y, C_CO2=c_co2)
    q = {
        'mu': kin_model.mu_current,
        'q_Glc': kin_model.q_Glc_current,
        'q_Gln': kin_model.q_Gln_current,
        'q_Lac': kin_model.q_Lac_current,
        'q_Amm': kin_model.q_Amm_current,
        'q_O2': kin_model.q_O2_current,
        'q_CO2': kin_model.q_CO2_current,
        'q_AA': kin_model.q_AA_current,
    }

    mu_eff = max(q['mu'], 1e-12)
    if sto_model.mode == StoichiometryMode.EXCESS_COUPLED and hasattr(kin_model, 'calculate_cat_yields_from_excess'):
        cat_yields = kin_model.calculate_cat_yields_from_excess(mu_eff, y[1], y[2])
        yld = sto_model.calculate_yields(mu_eff, cat_yields=cat_yields)
    else:
        yld = sto_model.calculate_yields(mu_eff)
    return q, yld

# Representative fed-batch-like state
Xv = 5.0e8
C_Glc_vals = np.linspace(0.5, 25.0, 80)
C_Gln = 1.5
C_Lac = 5.0
C_Amm = 1.0
V = 2.0
C_O2 = 0.21
C_CT = 1.5
C_H = 1e-7
C_Na = 0.0
C_AA = 5.0
t_eval = 24.0
C_CO2 = 0.05

rates = {'CONSTANT': {'q_Glc': [], 'q_Gln': [], 'q_Lac': [], 'q_O2': [], 'q_CO2': [], 'mu': []},
         'DYNAMIC': {'q_Glc': [], 'q_Gln': [], 'q_Lac': [], 'q_O2': [], 'q_CO2': [], 'mu': []},
         'EXCESS_COUPLED': {'q_Glc': [], 'q_Gln': [], 'q_Lac': [], 'q_O2': [], 'q_CO2': [], 'mu': []}}

for cg in C_Glc_vals:
    y = [Xv, cg, C_Gln, C_Lac, C_Amm, V, C_O2, C_CT, C_H, C_Na, C_AA]
    q_c, _ = eval_point(kin_const, sto_const, params_sto, t_eval, y, C_CO2)
    q_d, _ = eval_point(kin_dyn, sto_dyn, params_sto, t_eval, y, C_CO2)
    q_e, _ = eval_point(kin_exc_cmp, sto_exc, params_exc, t_eval, y, C_CO2)
    for key in ['q_Glc', 'q_Gln', 'q_Lac', 'q_O2', 'q_CO2', 'mu']:
        rates['CONSTANT'][key].append(q_c[key])
        rates['DYNAMIC'][key].append(q_d[key])
        rates['EXCESS_COUPLED'][key].append(q_e[key])

# Single-point stoich yield comparison at reference condition
y_ref = [Xv, 5.0, C_Gln, C_Lac, C_Amm, V, C_O2, C_CT, C_H, C_Na, C_AA]
q_c_ref, y_c = eval_point(kin_const, sto_const, params_sto, t_eval, y_ref, C_CO2)
q_d_ref, y_d = eval_point(kin_dyn, sto_dyn, params_sto, t_eval, y_ref, C_CO2)
q_e_ref, y_e = eval_point(kin_exc_cmp, sto_exc, params_exc, t_eval, y_ref, C_CO2)

yield_labels = [
    'Y_Glc/X', 'Y_Lac/Glc', 'Y_Gln/Glc', 'Y_O2/Glc', 'Y_CO2/Glc',
    'Y_AA/Glc', 'Y_Amm/Gln', 'Y_H2O/X',
]
df_yields = pd.DataFrame(
    [y_c, y_d, y_e],
    index=['CONSTANT', 'DYNAMIC', 'EXCESS_COUPLED'],
    columns=yield_labels,
)
display(df_yields.style.format('{:.4f}').background_gradient(cmap='YlGnBu', axis=0))

# Convert specific q-rates from mmol/cell/h to mmol per 10^9 cells/L/h
Q_SCALE = 1e9
q_metrics = ['q_Glc', 'q_Gln', 'q_Lac', 'q_O2', 'q_CO2']
plot_metrics = q_metrics + ['mu']
y_axis_labels = {
    'q_Glc': 'q_Glc (mmol/L/h at 10^9 cells/L)',
    'q_Gln': 'q_Gln (mmol/L/h at 10^9 cells/L)',
    'q_Lac': 'q_Lac (mmol/L/h at 10^9 cells/L)',
    'q_O2': 'q_O2 (mmol/L/h at 10^9 cells/L)',
    'q_CO2': 'q_CO2 (mmol/L/h at 10^9 cells/L)',
    'mu': 'mu (1/h)',
}

fig = make_subplots(rows=3, cols=2)
colors = {'CONSTANT': 'firebrick', 'DYNAMIC': 'royalblue', 'EXCESS_COUPLED': 'darkgreen'}

for i, metric in enumerate(plot_metrics):
    r = (i // 2) + 1
    c = (i % 2) + 1
    for model_name in ['CONSTANT', 'DYNAMIC', 'EXCESS_COUPLED']:
        y_vals = np.array(rates[model_name][metric])
        if metric in q_metrics:
            y_vals = y_vals * Q_SCALE

        fig.add_trace(
            go.Scatter(
                x=C_Glc_vals,
                y=y_vals,
                mode='lines',
                name=model_name if i == 0 else None,
                showlegend=(i == 0),
                line=dict(color=colors[model_name], width=2),
            ),
            row=r, col=c,
        )

    fig.update_yaxes(title_text=y_axis_labels[metric], row=r, col=c)

fig.update_xaxes(title_text='C_Glc (mmol/L)')
fig.update_layout(
    height=1200,
    width=1000,
    template='plotly_white',
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='center',
        x=0.5,
    ),
)
fig.show()

,Y_Glc/X,Y_Lac/Glc,Y_Gln/Glc,Y_O2/Glc,Y_CO2/Glc,Y_AA/Glc,Y_Amm/Gln,Y_H2O/X
CONSTANT,0.2575,0.5829,0.2934,2.8880,3.2020,0.2889,0.6550,0.9488
DYNAMIC,0.6531,0.7338,0.1987,3.6352,3.8005,0.1139,1.2172,2.4438
EXCESS_COUPLED,1.1871,1.6013,0.2947,1.9405,2.1242,0.0627,1.7096,1.8229
